# Retinexformer — training on LOL-v1 (Kaggle)

**Settings (right panel):** Accelerator = GPU (T4 x1 or P100), Internet = ON, Persistence = off.

**How to run it**
1. *First session:* click **Save Version → Save & Run All (Commit)**. You can close the browser; it runs for ≤ 10.5 h.
2. *Next session (resume):* open the notebook → **Add Input → Your Work/Notebooks** → add the *previous version's output* of
   this notebook. Cell 4 finds the old `experiments/` folder there and copies it back; Retinexformer's `train.py`
   then **auto-resumes** from the newest `.state` file. Commit again.

**Why `timeout 37800`?** Kaggle kills a session at 12 h, and a killed session saves **nothing**. Stopping ourselves
at 10.5 h lets the notebook finish normally, so the output (checkpoints + log) is kept.

In [ ]:
# 1. Check that we really have a GPU (should list a Tesla T4 or P100)
!nvidia-smi

In [ ]:
# 2. Get the code and install its dependencies. Must end with the line "basicsr import OK".
%cd /kaggle/working
!git clone -q https://github.com/caiyuanhao1998/Retinexformer.git
%cd /kaggle/working/Retinexformer
!pip install -q einops gdown addict future lmdb natsort yacs yapf lpips thop timm
# We do NOT need `python setup.py develop`: basicsr is a plain folder inside the repo,
# so we just tell Python to look in the repo folder (PYTHONPATH). Newer setuptools
# versions often fail on `setup.py develop`, which is why we avoid it.
import os, sys
os.environ["PYTHONPATH"] = "/kaggle/working/Retinexformer"   # used by the !python commands below
sys.path.insert(0, "/kaggle/working/Retinexformer")
!cd /kaggle/working/Retinexformer && python -c "from basicsr.models import create_model; from basicsr.data import create_dataset; print('basicsr import OK')"

In [ ]:
# 3. Data. The configs expect data/LOLv1/{Train,Test}/{input,target} relative to the repo.
#    Option A: a Kaggle dataset you uploaded that contains a LOLv1 folder -> we symlink it (no copy).
#    Option B: nothing attached -> download the same zip Retinexformer's README links to.
import glob, os, subprocess
os.makedirs("data", exist_ok=True)
found = glob.glob("/kaggle/input/**/LOLv1/Train/input", recursive=True)
if found:
    src = os.path.dirname(os.path.dirname(found[0]))          # .../LOLv1
    if not os.path.exists("data/LOLv1"):
        os.symlink(src, "data/LOLv1")
    print("Using attached dataset:", src)
else:
    print("No attached LOLv1 found -> downloading from Google Drive")
    for attempt in range(3):                                   # gdown is sometimes flaky; retry
        if subprocess.call("gdown -q 1L-kqSQyrmMueBh_ziWoPFhfsAh50h20H -O /kaggle/tmp_lolv1.zip", shell=True) == 0:
            break
    !mkdir -p /kaggle/tmp_lol && unzip -q -o /kaggle/tmp_lolv1.zip -d /kaggle/tmp_lol && ln -sfn /kaggle/tmp_lol/LOLv1 data/LOLv1
!echo "train: $(ls data/LOLv1/Train/input | wc -l) (expect 485)   test: $(ls data/LOLv1/Test/input | wc -l) (expect 15)"

In [ ]:
# 4. RESUME: if a previous version's output is attached, copy its experiments/ folder back.
#    train.py looks in experiments/<name>/training_states/ and resumes from the newest <iter>.state.
import glob, shutil
prev = glob.glob("/kaggle/input/**/Retinexformer/experiments/Enhancement_RetinexFormer_LOL_v1", recursive=True)
if prev:
    shutil.copytree(prev[0], "experiments/Enhancement_RetinexFormer_LOL_v1", dirs_exist_ok=True)
    states = sorted(glob.glob("experiments/Enhancement_RetinexFormer_LOL_v1/training_states/*.state"),
                    key=lambda p: int(os.path.basename(p).split(".")[0]))
    print("Resuming. Newest state:", states[-1] if states else "NONE (will start from scratch!)")
    # Keep the old log(s) under a different name so they are not lost
    for f in glob.glob("experiments/Enhancement_RetinexFormer_LOL_v1/train_*.log"):
        print("  previous log:", f)
else:
    print("No previous run attached -> training from scratch (iteration 0).")

In [ ]:
# 5. Small, Kaggle-specific edits to the config (printed below so you can see exactly what changed):
#    * num_worker_per_gpu 8 -> 4 : Kaggle gives 4 CPU cores; 8 loader workers just compete.
#    * save_checkpoint_freq 1000 -> 5000 : each training state (model + Adam optimizer) is saved;
#      every 1000 iters would produce ~150 states = several GB of output. 5000 is plenty.
#    Everything that affects the RESULT (lr, iterations, patch size, batch size, loss) is untouched.
import re
cfg = "Options/RetinexFormer_LOL_v1.yml"
s = open(cfg).read()
s = re.sub(r"num_worker_per_gpu: \d+", "num_worker_per_gpu: 4", s)
s = re.sub(r"save_checkpoint_freq: .*", "save_checkpoint_freq: !!float 5e3", s)
open(cfg, "w").write(s)
!grep -n -E "num_worker_per_gpu|save_checkpoint_freq|total_iter|batch_size_per_gpu|gt_size:|lr:" {cfg}

In [ ]:
# 6. TRAIN. 37800 s = 10.5 h. The log shows iterations, loss (l_pix) and validation PSNR every 1000 iters.
!PYTHONPATH=/kaggle/working/Retinexformer timeout 37800 python3 basicsr/train.py --opt Options/RetinexFormer_LOL_v1.yml 2>&1 | grep -v -i warning | tail -n 300

In [ ]:
# 7. Keep the output small: delete all but the 2 newest .state files (each holds the Adam optimizer too).
#    Model weights (models/net_g_*.pth, ~6 MB each) are all kept — we may want to evaluate several.
exp = "experiments/Enhancement_RetinexFormer_LOL_v1"
states = sorted(glob.glob(f"{exp}/training_states/*.state"), key=lambda p: int(os.path.basename(p).split(".")[0]))
for p in states[:-2]:
    os.remove(p)
print("kept states:", [os.path.basename(p) for p in states[-2:]])
print("models:", sorted(os.listdir(f"{exp}/models"))[-5:])

In [ ]:
# 8. Speed report -> how many sessions the full 150k-iteration schedule needs.
#    BasicSR log lines look like: [epoch: 3, iter: 2,000, lr:(3.000e-04,)] [eta: 1 day, ..., time (data): 0.250 (0.004)] l_pix: ...
import re
logs = sorted(glob.glob(f"{exp}/train_*.log"))
txt = open(logs[-1]).read()
iters = [int(x.replace(",", "")) for x in re.findall(r"iter:\s*([\d,]+)", txt)]
t_iter = [float(x) for x in re.findall(r"time \(data\): ([\d.]+)", txt)]
psnr = re.findall(r"psnr: ([\d.]+)", txt)
if iters and t_iter:
    sec = sum(t_iter[-20:]) / len(t_iter[-20:])
    print(f"reached iter {iters[-1]:,} | {sec:.3f} s/iter = {1/sec:.2f} it/s")
    per_session = 37800 / sec
    print(f"=> ~{per_session:,.0f} iters per 10.5 h session; 150,000 iters need ~{150000/per_session:.1f} sessions")
print("last validation PSNRs:", psnr[-5:])

In [ ]:
# 9. Copy the important small files to the top of /kaggle/working for easy download:
#    the log, the newest checkpoint, and the best-validation-PSNR checkpoint.
#    (net_g_latest.pth only appears when ALL 150k iterations are finished.)
os.makedirs("/kaggle/working/train_out", exist_ok=True)
for f in logs:
    shutil.copy(f, "/kaggle/working/train_out/")
models = sorted(glob.glob(f"{exp}/models/net_g_[0-9]*.pth"), key=lambda p: int(re.findall(r"(\d+)\.pth", p)[0]))
for f in models[-1:] + glob.glob(f"{exp}/models/best_psnr_*.pth") + glob.glob(f"{exp}/models/net_g_latest.pth"):
    shutil.copy(f, "/kaggle/working/train_out/")
!ls -la /kaggle/working/train_out